# M1 Notebook 02 — Sets, Logic, Relations, and Functions

**Book alignment:** *SRAI Book 1 — Mathematical Foundations*, Chapter 2  
**Production Unit:** PU-B01-C02  
**Notebook ID:** M1_N02  
**Status:** Controlled Colab-compatible final release v1.0.0  
**Random seed:** 42

> Sets identify objects, logic evaluates statements, relations connect entities, and functions formalize transformations.


## 1. Learning objectives

By the end of this notebook, the reader will be able to:

1. Perform core finite-set operations mathematically and in Python.
2. Construct and interpret truth tables.
3. Distinguish relations from functions.
4. Represent binary relations through pairs and matrices.
5. Implement predicates and rule-based filters.
6. Connect sets and logic to databases, knowledge graphs, AI, and Decision Intelligence.


## 2. Prerequisites

- Basic Python collections and functions
- Elementary algebra
- M1 Notebook 01: Mathematical Thinking and Reproducible Computation


## 3. Colab and local environment bootstrap

The following cell makes the controlled notebook runnable both from an installed local repository and from a fresh Google Colab runtime. In Colab, upload the complete PU-B01-C02 release ZIP when prompted.


In [ ]:
# Google Colab / local bootstrap
try:
    import srai_math
    print(f"srai_math is already available from: {srai_math.__file__}")

except ModuleNotFoundError:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "srai_math is unavailable. Run this notebook from the controlled "
            "PU-B01-C02 repository or open it in Google Colab and upload the release ZIP."
        ) from exc

    from pathlib import Path
    import importlib
    import subprocess
    import sys
    import zipfile

    print("Upload PU-B01-C02_GitHub_Repository_v1.0.zip")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    extract_dir = Path("/content/srai_release_b01_c02")
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_name) as archive:
        base = extract_dir.resolve()
        for member in archive.infolist():
            target = (extract_dir / member.filename).resolve()
            if target != base and base not in target.parents:
                raise ValueError(f"Unsafe ZIP member rejected: {member.filename}")
        archive.extractall(extract_dir)

    candidates = list(extract_dir.rglob("requirements-colab.txt"))
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one requirements-colab.txt, found {len(candidates)}."
        )
    repo_root = candidates[0].parent

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(repo_root / "requirements-colab.txt"),
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(repo_root),
    ])

    importlib.invalidate_caches()
    import srai_math
    print(f"srai_math installed from: {srai_math.__file__}")


## 3. Reproducibility and environment

In [ ]:
from srai_math.utils import environment_info, set_seed

SEED = 42
set_seed(SEED)
environment_info()


## 4. Mathematical intuition

A national registry contains citizens. A health registry contains diagnosed patients. A vaccination registry contains vaccinated people.

Questions such as:

- Who belongs to both the elderly and diabetic groups?
- Who belongs to at least one priority group?
- Who belongs to neither group?
- Which patient is linked to which clinic?

are questions about sets, Boolean logic, relations, and functions.


## 5. Formal definitions and notation

Let $A$ and $B$ be sets.

### Union

$$
A\cup B = \{x : x\in A \text{ or } x\in B\}
$$

### Intersection

$$
A\cap B = \{x : x\in A \text{ and } x\in B\}
$$

### Difference

$$
A\setminus B = \{x : x\in A \text{ and } x\notin B\}
$$

### Complement relative to a universe $U$

$$
A^c = U\setminus A
$$

### Cartesian product

$$
A \times B = \{(a,b) \mid a \in A,\; b \in B\}
$$


## 6. Manual worked example — Set operations

Let

$$
A=\{1,2,3,4\}, \qquad B=\{3,4,5,6\}.
$$

Then

$$
A\cup B=\{1,2,3,4,5,6\},
$$

$$
A\cap B=\{3,4\},
$$

$$
A\setminus B=\{1,2\}.
$$


In [ ]:
from srai_math.discrete import (
    cartesian_product,
    complement,
    difference,
    intersection,
    union,
)

A = {1, 2, 3, 4}
B = {3, 4, 5, 6}
U = set(range(1, 8))

results = {
    "union": union(A, B),
    "intersection": intersection(A, B),
    "difference_A_minus_B": difference(A, B),
    "complement_A": complement(A, U),
    "cartesian_product_small": cartesian_product({1, 2}, {"x", "y"}),
}
results


## 7. Numerical and logical verification

In [ ]:
assert results["union"] == {1, 2, 3, 4, 5, 6}
assert results["intersection"] == {3, 4}
assert results["difference_A_minus_B"] == {1, 2}
assert results["complement_A"] == {5, 6, 7}
assert len(results["cartesian_product_small"]) == 4

print("All set-operation verifications passed.")


## 8. Logic and truth tables

For propositions $P$ and $Q$:

$$
\neg P
$$

means NOT $P$,

$$
P\land Q
$$

means $P$ AND $Q$,

$$
P\lor Q
$$

means $P$ OR $Q$,

and material implication is

$$
P\rightarrow Q \equiv \neg P\lor Q.
$$


In [ ]:
import pandas as pd
from srai_math.discrete import implies, truth_table_2

and_table = pd.DataFrame(truth_table_2(lambda p, q: p and q))
or_table = pd.DataFrame(truth_table_2(lambda p, q: p or q))
implication_table = pd.DataFrame(truth_table_2(implies))

and_table, or_table, implication_table


### Interpretation of implication

The implication $P\rightarrow Q$ is false only when $P$ is true and $Q$ is false.

This is a formal logical relationship. It does **not** by itself establish causation.


In [ ]:
assert implies(True, False) is False
assert implies(True, True) is True
assert implies(False, True) is True
assert implies(False, False) is True

print("Material implication verified.")


## 9. De Morgan's laws

$$
\neg(P\land Q)\equiv(\neg P)\lor(\neg Q)
$$

and

$$
\neg(P\lor Q)\equiv(\neg P)\land(\neg Q).
$$


In [ ]:
rows = []
for p in (True, False):
    for q in (True, False):
        law_1_left = not (p and q)
        law_1_right = (not p) or (not q)

        law_2_left = not (p or q)
        law_2_right = (not p) and (not q)

        rows.append({
            "P": p,
            "Q": q,
            "not(P and Q)": law_1_left,
            "(not P) or (not Q)": law_1_right,
            "not(P or Q)": law_2_left,
            "(not P) and (not Q)": law_2_right,
        })

demorgan = pd.DataFrame(rows)
assert (demorgan["not(P and Q)"] == demorgan["(not P) or (not Q)"]).all()
assert (demorgan["not(P or Q)"] == demorgan["(not P) and (not Q)"]).all()
demorgan


## 10. Relations

A binary relation from $A$ to $B$ is any subset of the Cartesian product:

$$
R\subseteq A\times B.
$$

For example,

$$
R=\{(\text{Patient 1},\text{Clinic A}),
(\text{Patient 2},\text{Clinic B})\}.
$$


In [ ]:
from srai_math.discrete import (
    is_function,
    relation_domain,
    relation_range,
)

patient_clinic = {
    ("Patient 1", "Clinic A"),
    ("Patient 2", "Clinic B"),
    ("Patient 3", "Clinic A"),
}

relation_summary = {
    "domain": relation_domain(patient_clinic),
    "range": relation_range(patient_clinic),
    "is_function": is_function(patient_clinic),
}
relation_summary


## 11. Relation versus function

A relation $R\subseteq A\times B$ defines a total function $f:A\to B$ when every element of the declared domain $A$ maps to exactly one element of $B$. If only some elements of $A$ are mapped, the relation is a partial function.

The relation

$$
\{(a,1),(a,2)\}
$$

is not a function because the same input $a$ maps to two different outputs.


In [ ]:
valid_mapping = {("A", 1), ("B", 2), ("C", 3)}
invalid_mapping = {("A", 1), ("A", 2)}

assert is_function(valid_mapping)
assert not is_function(invalid_mapping)

print("Relation/function distinction verified.")


## 12. Relation matrices

For finite sets $A=\{a_1,\dots,a_m\}$ and $B=\{b_1,\dots,b_n\}$, a relation may be represented by a binary matrix

$$
M_{ij}=
\begin{cases}
1,&(a_i,b_j)\in R,\\
0,&\text{otherwise}.
\end{cases}
$$


In [ ]:
import numpy as np

patients = ["Patient 1", "Patient 2", "Patient 3"]
clinics = ["Clinic A", "Clinic B"]

matrix = np.zeros((len(patients), len(clinics)), dtype=int)

for i, patient in enumerate(patients):
    for j, clinic in enumerate(clinics):
        matrix[i, j] = int((patient, clinic) in patient_clinic)

relation_matrix = pd.DataFrame(matrix, index=patients, columns=clinics)
relation_matrix


## 13. Predicate logic as data filtering

A predicate is a Boolean-valued function. In data systems, predicates become filters.

Example:

> Select all people who are at least 65 years old and have diabetes.


In [ ]:
from srai_math.discrete import evaluate_predicate

people = [
    {"name": "Awa", "age": 70, "diabetes": True},
    {"name": "Moussa", "age": 68, "diabetes": False},
    {"name": "Fatou", "age": 61, "diabetes": True},
    {"name": "Ibrahima", "age": 75, "diabetes": True},
]

priority_people = evaluate_predicate(
    people, lambda person: person["age"] >= 65 and person["diabetes"]
)
priority_names = {person["name"] for person in priority_people}

assert priority_names == {"Awa", "Ibrahima"}
priority_names


## 14. Database interpretation

The following concepts map directly to common data operations:

| Mathematics | Database interpretation |
|---|---|
| Set | Table or result set |
| Membership | Row satisfies a condition |
| Intersection | Records satisfying both criteria |
| Union | Combined result sets |
| Difference | Exclusion |
| Cartesian product | All possible row pairs |
| Relation | Link between entities |
| Function | Deterministic mapping |
| Predicate | `WHERE` condition |


## 15. AI interpretation

- Expert systems encode logical rules.
- Knowledge graphs encode entities and relations.
- Search engines apply Boolean retrieval.
- Constraint solvers use predicates and relations.
- Neuro-symbolic systems combine statistical learning with formal logic.


## 16. Decision Intelligence interpretation

A Decision Intelligence system must define:

1. the relevant entities;
2. the admissible decision alternatives;
3. the constraints;
4. the logical eligibility rules;
5. the relationships among affected actors;
6. the transformation from evidence to recommendation.

Sets and logic therefore form the formal boundary of the decision problem.


## 17. SRAI application case — National vaccination prioritization

Suppose the policy defines priority eligibility as:

- age at least 65, **or**
- healthcare worker, **or**
- clinically vulnerable,

but excludes anyone already fully vaccinated.

This can be represented as a Boolean predicate over the citizen registry.


In [ ]:
citizens = [
    {"name": "Aminata", "age": 72, "health_worker": False, "vulnerable": False, "fully_vaccinated": False},
    {"name": "Cheikh", "age": 40, "health_worker": True, "vulnerable": False, "fully_vaccinated": False},
    {"name": "Mariama", "age": 50, "health_worker": False, "vulnerable": True, "fully_vaccinated": True},
    {"name": "Ousmane", "age": 45, "health_worker": False, "vulnerable": False, "fully_vaccinated": False},
]

def eligible(citizen):
    priority = (
        citizen["age"] >= 65
        or citizen["health_worker"]
        or citizen["vulnerable"]
    )
    return priority and not citizen["fully_vaccinated"]

eligible_citizens = [c["name"] for c in citizens if eligible(c)]
assert eligible_citizens == ["Aminata", "Cheikh"]
eligible_citizens


## 18. Complexity and engineering notes

For Python hash sets, membership, insertion, union, and intersection are typically efficient on average. However:

- Cartesian products grow as $O(|A||B|)$.
- Dense relation matrices require $O(|A||B|)$ memory.
- Sparse adjacency lists are preferable for large relations.
- Natural-language rules should not be assumed equivalent to formal predicates without validation.
- Logical rules must be versioned when public policy changes.


## 19. Common errors and diagnostics

- Confusing an element with a singleton set.
- Defining a complement without specifying the universe.
- Confusing disjoint events with independent events.
- Treating implication as causation.
- Assuming every relation is a function.
- Translating ambiguous policy language directly into code.
- Using a Cartesian product when a sparse relation would suffice.


## 20. Exercises

### Level A — Conceptual

1. Explain the difference between $x\in A$ and $\{x\}\subseteq A$.
2. Explain why a relation may fail to be a function.
3. Explain why implication does not prove causation.

### Level B — Mathematical

1. Verify both De Morgan laws using a complete truth table.
2. For $A=\{1,2,3\}$ and $B=\{a,b\}$, enumerate $A\times B$.
3. Determine whether $R=\{(1,a),(2,a),(3,b)\}$ is a function from $A$ to $B$.

### Level C — Computational

1. Implement symmetric difference from first principles.
2. Build a function that checks whether a finite relation is injective.
3. Convert a relation matrix back into a set of ordered pairs.

### Capstone

Create an eligibility engine for a public programme. Define the universe, eligibility sets, exclusion sets, relation to service centres, and a transparent explanation for every accepted or rejected case.


## 21. Complete solutions and validation guidance

### Level A — Conceptual

1. $x\in A$ states that $x$ is an element of $A$. By contrast, $\{x\}\subseteq A$ states that every element of the singleton set $\{x\}$—namely $x$—belongs to $A$. The statements are equivalent in truth value but refer to different objects.
2. A relation fails to be a function if an input maps to more than one output. It also fails to be a total function from a declared domain $A$ when any element of $A$ has no output.
3. Implication specifies a truth condition. It does not establish temporal order, a mechanism, control of confounding, or a causal intervention.

### Level B — Mathematical

1. Evaluate all four combinations of $P$ and $Q$. In every row, $\neg(P\land Q)=(\neg P)\lor(\neg Q)$ and $\neg(P\lor Q)=(\neg P)\land(\neg Q)$.
2. $A\times B=\{(1,a),(1,b),(2,a),(2,b),(3,a),(3,b)\}$.
3. $R=\{(1,a),(2,a),(3,b)\}$ is a total function from $A$ to $B$: every input in $A$ occurs exactly once. It is not injective because 1 and 2 share output $a$.

### Level C — Reference implementations

```python
def symmetric_difference_first_principles(a, b):
    return {x for x in a | b if (x in a) != (x in b)}

def is_injective_relation(relation, declared_domain):
    mapping = {}
    for x, y in relation:
        if x in mapping and mapping[x] != y:
            return False
        mapping[x] = y
    return set(mapping) == set(declared_domain) and len(set(mapping.values())) == len(mapping)

def matrix_to_relation(matrix, row_labels, column_labels):
    return {
        (row_labels[i], column_labels[j])
        for i in range(len(row_labels))
        for j in range(len(column_labels))
        if matrix[i][j] == 1
    }
```

The capstone solution must additionally document the universe, operational definitions, missing-value treatment, rule version, conflict precedence and a reason for every decision.


## 22. Research extensions

- Fuzzy logic for partial truth.
- Description logics and ontologies.
- Temporal logic for policies that change over time.
- Probabilistic logic for uncertain evidence.
- Neuro-symbolic AI.
- Formal verification of public-sector decision rules.


## 23. Key insight

Sets determine what belongs to the problem. Logic determines which statements are true. Relations determine what is connected. Functions determine how inputs map to outputs. Together, they form the formal language used by databases, algorithms, knowledge graphs, AI systems, and transparent Decision Intelligence.


In [ ]:
# Final controlled validation manifest
import platform
import sys

validation_manifest = {
    "production_unit": "PU-B01-C02",
    "notebook_id": "M1_N02",
    "seed": SEED,
    "python": platform.python_version(),
    "all_core_assertions_passed": True,
    "eligible_citizens": eligible_citizens,
    "set_union_size": len(results["union"]),
    "relation_matrix_shape": tuple(relation_matrix.shape),
}

assert validation_manifest["eligible_citizens"] == ["Aminata", "Cheikh"]
assert validation_manifest["set_union_size"] == 6
assert validation_manifest["relation_matrix_shape"] == (3, 2)
validation_manifest
